In [ ]:
import torch
import torch.nn as nn

# --------------------------------------------------------
# 1. Base Building Blocks
# --------------------------------------------------------

class PatchEmbed(nn.Module):
    """
    Splits the 224x224 image into 16x16 patches and projects them to the embedding dimension.
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # Using a Conv2d layer with stride=patch_size is the standard way to extract non-overlapping patches
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # Shape: (Batch, Embed_Dim, H/Patch, W/Patch)
        x = x.flatten(2).transpose(1, 2)  # Shape: (Batch, Num_Patches, Embed_Dim)
        return x

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

class Block(nn.Module):
    """
    Standard Transformer Block (LayerNorm -> Attention -> Add -> LayerNorm -> MLP -> Add)
    """
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
# --------------------------------------------------------
# 2. Encoder and Decoder Architectures
# --------------------------------------------------------

class MAE_Encoder(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=768, depth=12, num_heads=12):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embed.num_patches

        # Class token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim), requires_grad=False) 

        self.blocks = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, qkv_bias=True)
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, mask_indices=None):
        # We will handle the actual logic of passing ONLY the 25% visible patches 
        # inside the main MaskedAutoencoder wrapper class in the next step.
        pass # Placeholder for now to keep things modular

class MAE_Decoder(nn.Module):
    def __init__(self, num_patches=196, encoder_embed_dim=768, decoder_embed_dim=384, decoder_depth=12, decoder_num_heads=6):
        super().__init__()
        # Project from encoder dimension to decoder dimension
        self.decoder_embed = nn.Linear(encoder_embed_dim, decoder_embed_dim, bias=True)
        
        # Mask token representing the missing 75% patches
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        
        # Decoder positional embeddings
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim), requires_grad=False)

        self.blocks = nn.ModuleList([
            Block(dim=decoder_embed_dim, num_heads=decoder_num_heads, qkv_bias=True)
            for i in range(decoder_depth)
        ])
        self.norm = nn.LayerNorm(decoder_embed_dim)
        
        # Final linear layer to reconstruct the pixel patches (Patch Size: 16x16x3 = 768)
        self.decoder_pred = nn.Linear(decoder_embed_dim, 16**2 * 3, bias=True)

    def forward(self, x):
         # Logic for reconstructing patches will go here
         pass

In [ ]:
# --------------------------------------------------------
# 3. Main Masked Autoencoder Class
# --------------------------------------------------------

class MaskedAutoencoderViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=768, depth=12, num_heads=12,
                 decoder_embed_dim=384, decoder_depth=12, decoder_num_heads=6,
                 mlp_ratio=4., norm_layer=nn.LayerNorm):
        super().__init__()

        # --- ENCODER (ViT-Base) ---
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim), requires_grad=False)

        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias=True)
            for i in range(depth)])
        self.norm = norm_layer(embed_dim)

        # --- DECODER (ViT-Small) ---
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim), requires_grad=False)

        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_num_heads, mlp_ratio, qkv_bias=True)
            for i in range(decoder_depth)])

        self.decoder_norm = norm_layer(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * in_chans, bias=True)

    def patchify(self, imgs):
        """Split 224x224 image into 16x16 patches."""
        p = self.patch_embed.proj.kernel_size[0]
        h = w = imgs.shape[2] // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))
        return x

    def unpatchify(self, x):
        """Convert patches back to image space."""
        p = self.patch_embed.proj.kernel_size[0]
        h = w = int(x.shape[1]**.5)
        x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], 3, h * p, h * p))
        return imgs

    def random_masking(self, x, mask_ratio):
        """Randomly retain 25% patches and mask the remaining 75%."""
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        
        # Generate random noise to shuffle patches
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1) # Keep track to unshuffle later

        # Keep the first 25% visible patches
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))

        # Generate the binary mask (0 = visible, 1 = masked)
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :] # Add positional embeddings
        
        # Apply masking: retain 25%, drop 75%
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        # Process ONLY visible patches
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        
        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        # Project encoder output to decoder dimension
        x = self.decoder_embed(x)
        
        # Append learnable mask tokens
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
        x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)
        
        # Unshuffle tokens to maintain proper positional ordering for reconstruction
        x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))
        x = torch.cat([x[:, :1, :], x_], dim=1)
        
        # Add decoder positional embeddings
        x = x + self.decoder_pos_embed
        
        # Apply decoder blocks
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        
        # Reconstruct pixel patches
        x = self.decoder_pred(x)
        x = x[:, 1:, :] # Remove cls token prediction
        return x

    def forward(self, imgs, mask_ratio=0.75):
        # 1. Forward Pass through Encoder
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        
        # 2. Forward Pass through Decoder
        pred = self.forward_decoder(latent, ids_restore)
        
        return pred, mask

In [ ]:
import os
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
from torch.amp import GradScaler, autocast
import torch

# --------------------------------------------------------
# 4. Dataset Preparation
# --------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

dataset_path = '/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200/tiny-imagenet-200/train'
train_dataset = torchvision.datasets.ImageFolder(dataset_path, transform=transform)

# Using a small batch size (32-64) to fit Kaggle T4 x2 memory limits [cite: 81]
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

# --------------------------------------------------------
# 5. Custom Loss Function (Multi-GPU Ready)
# --------------------------------------------------------
def mae_loss(imgs, pred, mask, model):
    """
    Computes MSE loss only on the masked patches[cite: 72, 73].
    """
    base_model = model.module if hasattr(model, 'module') else model
    target = base_model.patchify(imgs)
    
    loss = (pred - target) ** 2
    loss = loss.mean(dim=-1) 
    
    # Apply mask: calculate loss ONLY on masked patches [cite: 74]
    loss = (loss * mask).sum() / mask.sum() 
    return loss

# --------------------------------------------------------
# 6. Training Loop Setup with Checkpoints (Multi-GPU Ready)
# --------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MaskedAutoencoderViT()

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

epochs = 60
start_epoch = 0
training_logs = []

# Optimizer & Scheduler [cite: 75, 76, 77, 78]
optimizer = optim.AdamW(model.parameters(), lr=1.5e-4, weight_decay=0.05) # [cite: 82]
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) 

scaler = GradScaler('cuda')

# --- RESUME FROM CHECKPOINT LOGIC ---
checkpoint_path = '/kaggle/working/mae_checkpoint_latest.pth'
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint at {checkpoint_path}. Resuming...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    training_logs = checkpoint.get('training_logs', [])
    print(f"Successfully loaded! Resuming from Epoch [{start_epoch + 1}/{epochs}]")
else:
    print("No checkpoint found. Starting Training from scratch...")

for epoch in range(start_epoch, epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (imgs, _) in enumerate(train_loader):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        
        with autocast('cuda'):
            pred, mask = model(imgs, mask_ratio=0.75)
            loss = mae_loss(imgs, pred, mask, model)
        
        scaler.scale(loss).backward()
        
        # Optional gradient clipping to prevent exploding gradients [cite: 83]
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    training_logs.append(avg_loss)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

    # --- SAVE CHECKPOINT LOGIC ---
    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'training_logs': training_logs
    }
    # Save/overwrite the latest checkpoint
    torch.save(checkpoint_data, checkpoint_path)
    
    # Save a permanent milestone checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(checkpoint_data, f'/kaggle/working/mae_checkpoint_epoch_{epoch+1}.pth')
        print(f"--> Saved milestone checkpoint for Epoch {epoch+1}")